# Engram: conditional memory as a second axis of sparsity

Toy simulations of the DeepSeek-linked Engram paper (arXiv:2601.07372):
lookup-vs-compute retrieval, the U-shaped sparsity-allocation law, freed attention
capacity for long context, and deterministic prefetch. All numbers from the paper
are reproduced as constants and checked with asserts.

## 1. Lookup vs compute: why a table beats a network at remembering

In [1]:
import numpy as np
rng = np.random.default_rng(0)

# Toy "knowledge base": 10,000 facts, each a 64-dim vector.
VOCAB, DIM = 10_000, 64
table = rng.normal(size=(VOCAB, DIM))

# Lookup: O(1) hash of the n-gram context -> exact vector.
def lookup(ids):
    return table[ids % VOCAB]

# Compute: a small 2-layer MLP must *reconstruct* the same vectors from an id embedding.
emb = rng.normal(size=(VOCAB, 16)) * 0.1
W1 = rng.normal(size=(16, 64)) * 0.1
def reconstruct(ids):
    return np.tanh(emb[ids % VOCAB] @ W1)

ids = rng.integers(0, VOCAB, size=2000)
err_lookup = np.mean((lookup(ids) - table[ids % VOCAB]) ** 2)
err_mlp = np.mean((reconstruct(ids) - table[ids % VOCAB]) ** 2)
print(f"lookup MSE: {err_lookup:.4f} (exact, by construction)")
print(f"untrained-MLP reconstruction MSE: {err_mlp:.4f}")
assert err_lookup == 0.0 and err_mlp > 1.0
print("Point: a lookup returns the fact; a network must spend parameters re-deriving it.")

lookup MSE: 0.0000 (exact, by construction)
untrained-MLP reconstruction MSE: 1.0022
Point: a lookup returns the fact; a network must spend parameters re-deriving it.


## 2. The parameter budget: 72 experts -> 55 experts + 5.7B lookup

In [1]:
# Paper's Engram-27B config (iso-parameter, iso-FLOPs vs MoE-27B).
moe_experts, engram_experts = 72, 55
engram_memory_B = 5.7
total_B, activated_B = 26.7, 3.8
print(f"experts removed: {moe_experts - engram_experts} (reallocated to lookup memory)")
print(f"total {total_B}B, activated {activated_B}B -> activation ratio {activated_B/total_B:.1%}")
assert moe_experts - engram_experts == 17
assert abs(activated_B / total_B - 0.142) < 0.01  # ~14% of params fire per token

experts removed: 17 (reallocated to lookup memory)
total 26.7B, activated 3.8B -> activation ratio 14.2%


## 3. The U-shaped Sparsity Allocation law (toy)

In [1]:
# Toy loss: memory helps up to a point, then starves the reasoning backbone.
# loss(m) = base + a*(m - m*)^2, m = memory share of the sparse budget.
m = np.linspace(0, 1, 101)
m_star, a, base = 0.21, 0.35, 1.0
loss = base + a * (m - m_star) ** 2
opt = m[np.argmin(loss)]
print(f"toy optimum memory share: {opt:.2f} (interior, not 0 or 1)")
assert 0.05 < opt < 0.95, "optimum must be interior"
assert loss[0] > loss.min() and loss[-1] > loss.min()
print("All-memory is as bad as no-memory: the backbone still has to reason.")

toy optimum memory share: 0.21 (interior, not 0 or 1)
All-memory is as bad as no-memory: the backbone still has to reason.


## 4. Freed attention: long-context retrieval 84.2 -> 97.0

In [1]:
# Paper's Multi-Query NIAH @ 32k: MoE-27B 84.2, Engram-27B (50k) 97.0.
# Toy mechanism: attention has a fixed budget; local dependencies eat a share.
# Offloading locals to lookup returns that share to global retrieval.
def retrieval_accuracy(local_share):
    # accuracy rises as the share spent on local reconstruction falls
    return 0.60 + 0.40 * (1 - local_share)

moe_local, engram_local = 0.40, 0.08  # illustrative shares
acc_moe, acc_engram = retrieval_accuracy(moe_local), retrieval_accuracy(engram_local)
print(f"toy: MoE {acc_moe:.2f} -> Engram {acc_engram:.2f}")
print("paper: MoE 84.2 -> Engram 97.0 (MQ-NIAH @ 32k)")
assert acc_engram > acc_moe
assert abs(97.0 - 84.2 - 12.8) < 1e-9

toy: MoE 0.84 -> Engram 0.97
paper: MoE 84.2 -> Engram 97.0 (MQ-NIAH @ 32k)


## 5. Deterministic prefetch: why the lookup costs ~nothing

In [1]:
# MoE routing is data-dependent (can't prefetch); Engram addressing is a
# deterministic function of the token n-gram, so entries prefetch from host memory.
# Latency model: lookup time overlapped with compute when prefetch hides it.
compute_ms, lookup_ms, overlap = 10.0, 8.0, 0.95
effective_lookup = lookup_ms * (1 - overlap)  # only the unhidden tail shows up
total_moe = compute_ms
total_engram = compute_ms + effective_lookup
print(f"per-token time: MoE {total_moe:.2f} ms, Engram {total_engram:.2f} ms "
      f"(+{100*(total_engram/total_moe-1):.0f}% overhead)")
assert total_engram < total_moe * 1.10  # paper: "negligible overhead"
print("Deterministic addressing is the whole engineering trick: prefetchable memory is free memory.")

per-token time: MoE 10.00 ms, Engram 10.40 ms (+4% overhead)
Deterministic addressing is the whole engineering trick: prefetchable memory is free memory.
